# Nemotron-3-Nano SAE Workflow

This notebook demonstrates the complete workflow for:
1. Training a sparse autoencoder on Nemotron-3-Nano activations
2. Running auto-interpretation to generate feature descriptions
3. Exporting data for the interactive feature viewer

**Model**: `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16` (30B params, 3B active)
- Hybrid architecture: Mamba-2 / MoE / GQA across 52 layers
- Hidden dim: 2688
- Requires `trust_remote_code=True`, bf16 precision, `device_map="auto"`

**Data**: FineWeb (sample-10BT subset) via streaming

### Prerequisites (CUDA required)

Nemotron's Mamba-2 layers need CUDA-only packages. Install on your GPU server:

```bash
pip install mamba-ssm causal-conv1d
```

This notebook **must run on a machine with NVIDIA GPUs**.

In [1]:
import json
from pathlib import Path

import torch

# Nemotron-specific imports
from nemotron_sae import NemotronModel, load_fineweb
from sae import (
    AutoInterpreter,
    TokenActivationCollector,
    build_cluster_label_prompt,
    compute_cluster_centroids,
    save_cluster_labels,
)
from sae.activation_store import load_activations
from sae.analysis import compute_feature_logits

# SAE imports
from sae.architectures import TopKSAE
from sae.autointerp import (
    AnthropicClient,
    NIMClient,
    NVIDIAInternalClient,
    OpenAIClient,
)
from sae.training import Trainer, TrainingConfig
from sae.utils import get_device, set_seed
from tqdm import tqdm

/Users/jwilber/Desktop/biosae_refactor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration
set_seed(42)
DEVICE = get_device()
print(f"Using device: {DEVICE}")

# Paths
OUTPUT_DIR = Path("./outputs_nemotron")
OUTPUT_DIR.mkdir(exist_ok=True)
ACTIVATION_STORE_PATH = OUTPUT_DIR / "activations_store"

Using device: mps


## 1. Load Data and Extract Nemotron Activations

FineWeb is streamed from HuggingFace (no full download needed).
The 30B model uses `device_map="auto"` for multi-GPU sharding and bf16 precision.

In [3]:
# Load text data via streaming
import random


texts = load_fineweb(
    split="train",
    max_samples=20_000,
    min_length=30,
    subset="sample-10BT",
)

# Shuffle for diversity
random.shuffle(texts)
print(f"Loaded {len(texts)} texts (shuffled)")

Loaded 20000 texts from FineWeb (sample-10BT) train
Loaded 20000 texts (shuffled)


In [4]:
texts = texts[0:5_000]

In [5]:
# Load Nemotron model (bf16, auto device map)
# This loads the full CausalLM — reused later for loss_recovered eval
nemotron = NemotronModel(
    model_name="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16",
    layer=39,  # 3/4 depth (52 layers total)
    max_length=2048,
)
print(f"Hidden dim: {nemotron.hidden_size}")

A new version of the following files was downloaded from https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16:
- configuration_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!
A new version of the following files was downloaded from https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16:
- modeling_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ImportError: mamba-ssm is required by the Mamba model but cannot be imported

In [ ]:
# Extract activations in chunks and stream to disk (memory efficient)
# This avoids loading all embeddings into memory at once

from sae import ActivationStore


CHUNK_SIZE = 50  # Smaller chunks for 30B model

store = ActivationStore(ACTIVATION_STORE_PATH)
tokens_per_text = []  # Track sequence boundaries

for i in tqdm(range(0, len(texts), CHUNK_SIZE), desc="Extracting"):
    chunk_texts = texts[i : i + CHUNK_SIZE]

    # Extract activations for this chunk (batch_size=4 for 30B model)
    embeddings, masks = nemotron.generate_activations(chunk_texts, batch_size=4, show_progress=False)

    # Track tokens per text for sequence boundary reconstruction
    tokens_per_text.extend(masks.sum(dim=1).tolist())

    # Flatten and append to store
    valid_mask = masks.bool()
    embeddings_flat = embeddings[valid_mask]
    store.append(embeddings_flat)

    # Memory freed each iteration
    del embeddings, masks, embeddings_flat

# Finalize store with metadata including sequence boundaries
store.finalize(
    metadata={
        "model": "nemotron-3-nano",
        "layer": nemotron.layer,
        "n_texts": len(texts),
        "tokens_per_text": tokens_per_text,
    }
)

print(f"Total tokens: {sum(tokens_per_text):,}")

## 2. Train Sparse Autoencoder

In [ ]:
# Load activation store (can skip extraction if already saved)
store = load_activations(ACTIVATION_STORE_PATH)
print(f"Loaded {store.n_samples:,} activations, dim={store.hidden_dim}")

# Create SAE with input normalization for stable training
INPUT_DIM = store.hidden_dim  # 2688 for Nemotron
EXPANSION_FACTOR = 16
HIDDEN_DIM = INPUT_DIM * EXPANSION_FACTOR
TOP_K = 32

sae = TopKSAE(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    top_k=TOP_K,
    normalize_input=True,
    auxk=32,
    auxk_coef=1 / 32,
    dead_tokens_threshold=100000,
)

# Initialize pre_bias from data (geometric median for robust centering)
sample_loader = store.get_dataloader(batch_size=2048, shuffle=True)
sample_batch = next(iter(sample_loader))
sae.init_pre_bias_from_data(sample_batch)

print(f"SAE: {INPUT_DIM} -> {HIDDEN_DIM} (top-{TOP_K}, normalized)")

In [ ]:
from sae.perf_logger import PerfLogger


# Create performance logger
perf_logger = PerfLogger(
    log_interval=150,
    use_wandb=False,
    print_logs=True,
    device=DEVICE,
)

In [ ]:
# Get dataloader from store
BATCH_SIZE = 512
dataloader = store.get_dataloader(batch_size=BATCH_SIZE, shuffle=True)

# Train
config = TrainingConfig(lr=3e-4, n_epochs=1, batch_size=BATCH_SIZE, device=DEVICE, log_interval=1500)

trainer = Trainer(sae, config, perf_logger=perf_logger)
final_loss = trainer.fit(dataloader)
perf_logger.finish()
print(f"\nFinal loss: {final_loss:.4f}")

In [ ]:
# Save checkpoint
checkpoint = {
    "model_state_dict": sae.state_dict(),
    "input_dim": INPUT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "model_config": {"type": "topk", "top_k": TOP_K},
}
torch.save(checkpoint, OUTPUT_DIR / "sae_checkpoint.pt")
print(f"Saved checkpoint to {OUTPUT_DIR / 'sae_checkpoint.pt'}")

## 2b. Post-Training Evaluation

Run the standard evaluation suite: reconstruction quality, sparsity statistics, and loss recovered.

**Note**: Loss recovered reuses the CausalLM already loaded in `nemotron.model` — no second 30B model load needed.

In [ ]:
from nemotron_sae.eval import evaluate_nemotron_loss_recovered
from sae.eval import evaluate_sae


# Load fresh eval data (streaming gives different samples)
eval_texts = load_fineweb(split="train", max_samples=200, min_length=30, subset="sample-10BT")
eval_embeddings, eval_masks = nemotron.generate_activations(eval_texts, batch_size=4)
eval_embeddings_flat = eval_embeddings[eval_masks.bool()]
print(f"Eval on {eval_embeddings_flat.shape[0]:,} tokens")

# Loss recovered — reuses nemotron.model (already loaded CausalLM)
layer_idx = nemotron.layer

# Run evaluation
results = evaluate_sae(
    sae,
    eval_embeddings_flat,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    loss_recovered_fn=lambda: evaluate_nemotron_loss_recovered(
        sae=sae,
        model=nemotron.model,
        tokenizer=nemotron.tokenizer,
        texts=eval_texts[:100],
        layer_idx=layer_idx,
        device=DEVICE,
    ),
)
results.print_summary()
results.save(OUTPUT_DIR / "eval_results.json")

# Cleanup eval tensors to free memory
del eval_embeddings, eval_masks, eval_embeddings_flat

## 3. Collect Feature Statistics and Token Examples

Using `TokenActivationCollector` from the library to compute:
1. **Per-feature stats** (frequency, mean, max, std) — vectorized torch ops
2. **Text-level activations** (`text_codes`) — max-pooled across tokens per text
3. **Top-k token examples** per feature — bounded min-heaps (memory efficient)

All in a single pass, using only O(n_features x top_k) memory.

In [ ]:
sae.eval()
n_features = sae.hidden_dim


# Define the encode function: takes a text, returns (token_labels, sae_codes)
def nemotron_encode(text):
    encoding = nemotron.tokenizer(text, max_length=nemotron.max_length, truncation=True, return_tensors="pt")
    input_ids = encoding["input_ids"].to(nemotron.model.device)
    attention_mask = encoding["attention_mask"].to(nemotron.model.device)

    with torch.no_grad():
        hidden_states, _ = nemotron.forward_features(input_ids, attention_mask)
        # Cast to float32 for SAE (model outputs bf16)
        codes = sae.encode(hidden_states.float().squeeze(0))

    tokens = [nemotron.tokenizer.decode([t]) for t in input_ids[0].tolist()]
    n_tokens = attention_mask.sum().item()
    return tokens[:n_tokens], codes[:n_tokens].cpu()


# Collect stats and top-k token examples in one pass
collector = TokenActivationCollector(
    nemotron_encode,
    n_features=n_features,
    top_k_tokens=50,
    store_token_codes=True,  # needed for features.json export and counter-examples
)
result = collector.collect(texts)

print(f"Processed {result.total_tokens:,} tokens across {result.total_texts} texts")
print(
    f"Feature frequency range: {min(s.activation_freq for s in result.feature_stats):.4f} - {max(s.activation_freq for s in result.feature_stats):.4f}"
)
print(
    f"Max activation range: {min(s.max_activation for s in result.feature_stats):.2f} - {max(s.max_activation for s in result.feature_stats):.2f}"
)

In [ ]:
# Analyze feature frequency distribution
freqs = torch.tensor([s.activation_freq for s in result.feature_stats])

print("Feature frequency distribution:")
print(f"  Min: {freqs.min():.4f}")
print(f"  Max: {freqs.max():.4f}")
print(f"  Mean: {freqs.mean():.4f}")
print(f"  Median: {freqs.median():.4f}")

# Count features at different thresholds
thresholds = [0.0, 0.001, 0.01, 0.05, 0.1]
print("\nFeatures above threshold:")
for t in thresholds:
    n = (freqs > t).sum().item()
    print(f"  >{t:.3f}: {n} features ({100 * n / HIDDEN_DIM:.1f}%)")

In [ ]:
# ===== INTERPRETATION CONFIG =====
MIN_FREQ_THRESHOLD = 0.01  # Features must activate on >1% of samples
MAX_FEATURES = 50  # Set to limit (e.g., 100), or None for all above threshold
# ==================================

# Select features above threshold
mask = freqs > MIN_FREQ_THRESHOLD
candidate_indices = torch.where(mask)[0]

# Optionally limit to top N by frequency
if MAX_FEATURES and len(candidate_indices) > MAX_FEATURES:
    candidate_freqs = freqs[candidate_indices]
    top_k = torch.argsort(candidate_freqs, descending=True)[:MAX_FEATURES]
    top_feature_indices = candidate_indices[top_k].tolist()
    print(f"Selected top {MAX_FEATURES} features (from {len(candidate_indices)} above threshold)")
else:
    top_feature_indices = candidate_indices.tolist()
    print(f"Selected {len(top_feature_indices)} features with freq > {MIN_FREQ_THRESHOLD}")

print(f"Feature indices sample: {top_feature_indices[:10]}...")

## 4. Auto-Interpretation with LLM

**Token-level approach**: Instead of showing the LLM full text passages, we show:
1. **Top activating tokens** with their activation magnitudes
2. **Context windows** around those tokens
3. **Decoder logit evidence** (which tokens the feature promotes/suppresses)
4. **Non-activating examples** as controls

This requires collecting token-level activations first.

In [ ]:
# Token-level data already collected by TokenActivationCollector above
# result.token_examples has top-k activating tokens per feature
# result.get_text_labels(idx) returns token strings for context windows
# result.get_text_codes(idx) returns full token codes (store_token_codes=True)

n_features_with_examples = len(result.token_examples)
total_examples = sum(len(v) for v in result.token_examples.values())
print(f"Token examples: {total_examples:,} across {n_features_with_examples} features")
print(f"Text labels stored for {result.total_texts} texts")

In [ ]:
# Compute decoder logits (which tokens each feature promotes/suppresses)
# For Nemotron, the output projection is lm_head
print("Computing feature logits from decoder weights...")

# Get unembedding matrix (lm_head weights, or tied embed_tokens)
if hasattr(nemotron.model, "lm_head"):
    unembedding = nemotron.model.lm_head.weight.detach().float()
else:
    unembedding = nemotron.model.model.embed_tokens.weight.detach().float()

vocab = [nemotron.tokenizer.decode([i]) for i in range(len(nemotron.tokenizer))]
feature_logits = compute_feature_logits(sae.to(unembedding.device), unembedding, vocab, top_k=10)
logits_by_feature = {fl.feature_id: fl for fl in feature_logits}

print(f"Computed logits for {len(feature_logits)} features")

In [ ]:
# Text-level activations computed by collector (max-pooled across tokens)
print(f"Text-level activations: {result.text_codes.shape}")
print(f"Number of texts: {result.total_texts}")

In [ ]:
import os


# ===== AUTO-INTERPRETATION USING LIBRARY =====
# Uses AutoInterpreter with collector-based token-level prompts

# ===== LLM CONFIGURATION =====
LLM_PROVIDER = "nvidia_internal"  # "anthropic", "openai", "nim", or "nvidia_internal"
LLM_MODELS = {
    "anthropic": "claude-sonnet-4-20250514",
    "openai": "gpt-4o",
    "nim": "nvidia/nemotron-mini-4b-instruct",
    "nvidia_internal": "aws/anthropic/bedrock-claude-3-7-sonnet-v1",
}


def get_llm_client(provider):
    """Get LLM client for the specified provider."""
    if provider == "anthropic":
        return AnthropicClient(model=LLM_MODELS["anthropic"])
    elif provider == "openai":
        return OpenAIClient(model=LLM_MODELS["openai"])
    elif provider == "nim":
        return NIMClient(model=LLM_MODELS["nim"])
    elif provider == "nvidia_internal":
        return NVIDIAInternalClient(model=LLM_MODELS["nvidia_internal"])
    else:
        raise ValueError(f"Unknown provider: {provider}")


env_var_map = {
    "anthropic": "ANTHROPIC_API_KEY",
    "openai": "OPENAI_API_KEY",
    "nim": "NIM_API_KEY",
    "nvidia_internal": "CLAUDE_SONNET_INFERENCE_API_KEY",
}
HAS_API_KEY = bool(os.environ.get(env_var_map[LLM_PROVIDER]))

if HAS_API_KEY:
    client = get_llm_client(LLM_PROVIDER)
    print(f"Using {LLM_PROVIDER} with model: {LLM_MODELS[LLM_PROVIDER]}")

    # Build logits map for decoder weight evidence
    logits_map = {fl.feature_id: fl for fl in feature_logits}

    # Run interpretation using library AutoInterpreter with collector
    interpreter = AutoInterpreter(llm_client=client, max_workers=5)
    interp_results = interpreter.interpret_features(
        collector=result,
        logits=logits_map,
        feature_indices=top_feature_indices,
        n_examples=5,
        context_window=3,
    )

    # Convert to dict format for backward compatibility
    results = [{"feature_idx": r.feature_idx, "description": r.description, "model": r.model} for r in interp_results]

    # Save results
    interpreter.save_results(interp_results, OUTPUT_DIR / "interpretations.json")

    # Show samples
    print("\nSample interpretations:")
    for r in results[:5]:
        print(f"  Feature {r['feature_idx']}: {r['description'][:80]}...")
else:
    print(f"No {env_var_map[LLM_PROVIDER]} found. Skipping auto-interpretation.")
    results = []

## 5. UMAP Visualization

Compute UMAP embedding of SAE decoder weights for feature exploration.

In [ ]:
from sae.analysis import compute_feature_umap, save_feature_atlas


# Compute UMAP from decoder weights
print("Computing UMAP embedding from decoder weights...")
geometry = compute_feature_umap(sae, n_neighbors=15, min_dist=0.1, random_state=42)
print(f"UMAP computed for {len(geometry.feature_ids)} features")

# Build labels from interpretations (use feature index if no interpretation)
descriptions = {}
if results:
    for r in results:
        feat_idx = r["feature_idx"] if isinstance(r, dict) else r.feature_idx
        desc = r["description"] if isinstance(r, dict) else r.description
        descriptions[feat_idx] = desc

labels = []
for i in range(HIDDEN_DIM):
    if i in descriptions:
        labels.append(descriptions[i])
    else:
        labels.append(f"Feature {i}")

In [ ]:
# Save feature atlas for dashboard (uses stats from collector directly)
save_feature_atlas(
    stats=result.feature_stats,
    geometry=geometry,
    output_path=OUTPUT_DIR / "feature_atlas.parquet",
    labels=labels,
)
print(f"Saved feature atlas to {OUTPUT_DIR / 'feature_atlas.parquet'}")

### Cluster Labels (optional)

If HDBSCAN clusters were computed, generate short labels for each cluster using the LLM.

In [ ]:
# Generate cluster labels using LLM (requires clusters from HDBSCAN)
clusters = compute_cluster_centroids(geometry)
print(f"Found {len(clusters)} clusters")

if clusters and HAS_API_KEY:
    # Build prompts for each cluster
    prompts = [build_cluster_label_prompt(c, descriptions, result.feature_stats) for c in clusters]

    # Generate labels
    print(f"Generating labels for {len(clusters)} clusters...")
    responses = client.generate_batch(prompts, max_workers=5, show_progress=True)
    cluster_labels = [r.text.strip() for r in responses]

    # Save cluster labels JSON for dashboard
    save_cluster_labels(clusters, cluster_labels, OUTPUT_DIR / "cluster_labels.json")

    # Show samples
    print("\nSample cluster labels:")
    for c, label in zip(clusters[:10], cluster_labels[:10]):
        print(f"  Cluster {c.cluster_id} ({c.size} features): {label}")
elif not clusters:
    print("No clusters found (install hdbscan for clustering)")
else:
    print("No API key - skipping cluster labeling")

## 6. Export Data for Feature Viewer

In [ ]:
# Token data already collected by TokenActivationCollector in Section 3
# No need for separate token collection - result has everything
print(f"Token data ready: {result.total_texts} texts, {result.total_tokens:,} tokens")

In [ ]:
# Build features.json using collector data
N_EXAMPLES = 5
export_feature_indices = sorted(range(n_features), key=lambda i: -result.feature_stats[i].activation_freq)

logits_map = {fl.feature_id: fl for fl in feature_logits}

feature_data = []
for feature_idx in tqdm(export_feature_indices, desc="Building features"):
    stats = result.feature_stats[feature_idx]

    # Get top activating texts from text_codes
    text_acts = result.text_codes[:, feature_idx]
    top_text_indices = torch.argsort(text_acts, descending=True)

    # Build examples from top texts
    examples = []
    for text_idx in top_text_indices[:N_EXAMPLES]:
        text_idx = text_idx.item()
        max_act = text_acts[text_idx].item()
        if max_act <= 0:
            break
        text_labels = result.get_text_labels(text_idx)
        text_codes = result.get_text_codes(text_idx)

        if text_codes is not None:
            acts = text_codes[:, feature_idx]
            token_data = [{"token": tok, "activation": act.item()} for tok, act in zip(text_labels, acts)]
        else:
            token_data = [{"token": tok, "activation": 0.0} for tok in text_labels]

        examples.append(
            {
                "text_idx": text_idx,
                "max_activation": max_act,
                "tokens": token_data,
            }
        )

    # Get logits
    fl = logits_map.get(feature_idx)
    top_positive = [[t, round(v, 3)] for t, v in fl.top_positive] if fl else []
    top_negative = [[t, round(v, 3)] for t, v in fl.top_negative] if fl else []

    feature_data.append(
        {
            "feature_id": feature_idx,
            "description": descriptions.get(feature_idx, f"Feature {feature_idx}"),
            "activation_freq": stats.activation_freq,
            "max_activation": stats.max_activation,
            "top_positive_logits": top_positive,
            "top_negative_logits": top_negative,
            "examples": examples,
        }
    )

In [ ]:
# Save features.json (same directory as parquet for launch_dashboard)
output = {
    "model": "nemotron-3-nano",
    "layer": nemotron.layer,
    "sae_hidden_dim": HIDDEN_DIM,
    "n_texts": len(texts),
    "features": feature_data,
}

with open(OUTPUT_DIR / "features.json", "w") as f:
    json.dump(output, f)

file_size = (OUTPUT_DIR / "features.json").stat().st_size / 1024
print(f"Saved {len(feature_data)} features to {OUTPUT_DIR / 'features.json'}")
print(f"File size: {file_size:.1f} KB")

## 7. Launch Interactive Dashboard

Launch the UMAP feature explorer with crossfiltering, histograms, and token-level highlighting.

In [ ]:
from sae import launch_dashboard


print("To launch the interactive UMAP dashboard:")
print()
print("  from sae import launch_dashboard")
print(f"  proc = launch_dashboard('{OUTPUT_DIR / 'feature_atlas.parquet'}')")
print("  # proc.terminate() to stop")
print()
print("The dashboard is bundled in the sae package at sae/src/sae/dashboard/")

In [ ]:
# Launch the interactive UMAP dashboard
# This opens a browser with the full visualization

LAUNCH_VIEWER = True  # Set to True to auto-launch

if LAUNCH_VIEWER:
    proc = launch_dashboard(OUTPUT_DIR / "feature_atlas.parquet")
    # proc.terminate() to stop the server

## Summary

This notebook demonstrated:

1. **Data Loading**: Used FineWeb (sample-10BT) via `load_fineweb()` with streaming
2. **Activation Extraction**: Used `NemotronModel` to extract layer 39 activations (bf16, multi-GPU), saved to `ActivationStore`
3. **SAE Training**: Loaded from store, trained TopK SAE with 16x expansion
4. **Post-Training Eval**: Used `evaluate_sae()` for reconstruction, sparsity, and loss recovered (reusing already-loaded CausalLM)
5. **Feature Collection**: Used `TokenActivationCollector` for memory-efficient single-pass stats + token examples
6. **Auto-Interpretation**: Used `AutoInterpreter` with collector-based token-level prompts
   - Supports **Anthropic** (Claude), **OpenAI** (GPT-4), **NVIDIA NIM**, and **NVIDIA Internal** APIs
7. **UMAP Visualization**: Computed 2D embedding of decoder weights and saved `feature_atlas.parquet`
8. **Cluster Labels**: Generated short labels for UMAP clusters using LLM (optional, requires HDBSCAN)
9. **Feature Viewer Export**: Generated `features.json` with token-level activations

**Output files:**
```
outputs_nemotron/
├── activations_store/       # Sharded parquet activations
├── sae_checkpoint.pt        # Trained SAE weights
├── eval_results.json        # Post-training evaluation metrics
├── interpretations.json     # LLM feature descriptions
├── feature_atlas.parquet    # UMAP + stats for dashboard
├── cluster_labels.json      # Cluster labels for dashboard (optional)
└── features.json            # Token-level activations for viewer
```

**Launch the dashboard:**
```python
from sae import launch_dashboard
proc = launch_dashboard("outputs_nemotron/feature_atlas.parquet")
# proc.terminate() to stop
```

The dashboard provides:
- Interactive UMAP embedding with zoom/pan
- Crossfiltering across UMAP, histograms, and search
- Feature cards with token-level activation highlighting
- Cluster labels on the UMAP (when cluster_labels.json is available)